In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 70.9 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26n.pt')

results = model.train(
    data='/content/drive/MyDrive/Projeto_Final/dataset/data.yaml',
    epochs=100,
    patience=20,
    imgsz=416,

    batch=8,
    project='/content/drive/MyDrive/Projeto_Final/treinamentos',
    name='teste_transistores',


    degrees=10,          # pequena rotação aleatória
    translate=0.1,       # desloca a imagem levemente
    scale=0.3,           # varia o zoom
    shear=5,             # leve distorção angular
    fliplr=0.5,          # espelha horizontalmente
    flipud=0.0,
    hsv_h=0.015,         # variação de matiz
    hsv_s=0.3,           # variação de saturação
    hsv_v=0.3,           # variação de brilho
    mosaic=1.0,          # combina 4 imagens em 1
    mixup=0.1,           # mistura duas imagens
    copy_paste=0.1,      # copia objetos de uma imagem para outra
)

# Validação formal
metrics = model.val()
print(metrics.box.map)     # mAP50-95
print(metrics.box.map50)   # mAP50
print(metrics.box.mp)      # precision média
print(metrics.box.mr)      # recall médio

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.88 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Projeto_Final/dataset/data.yaml, degrees=10, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None,

In [4]:
!pip install gradio ultralytics -q

import gradio as gr
from ultralytics import YOLO
import cv2

caminho_modelo = '/content/drive/MyDrive/Projeto_Final/treinamentos/teste_transistores/weights/best.pt'
modelo = YOLO(caminho_modelo)

def inspecionar_transistor(imagem):
    resultados = modelo(imagem, conf=0.15)  # limiar de confiança

    imagem_anotada = resultados[0].plot()
    imagem_anotada_rgb = cv2.cvtColor(imagem_anotada, cv2.COLOR_BGR2RGB)

    deteccoes = resultados[0].boxes
    if len(deteccoes) == 0:
        texto = "Nenhum objeto detectado."
    else:
        nomes = [modelo.names[int(c)] for c in deteccoes.cls]
        confs = [f"{float(c)*100:.1f}%" for c in deteccoes.conf]
        texto = "\n".join([f"{n}: {c}" for n, c in zip(nomes, confs)])

    return imagem_anotada_rgb, texto

# 4. Interface visual
interface = gr.Interface(
    fn=inspecionar_transistor,
    inputs=gr.Image(type="numpy", label="1. Suba a foto do Transistor aqui"),
    outputs=[
        gr.Image(type="numpy", label="2. Diagnóstico da IA"),
        gr.Textbox(label="3. Detecções e confiança")
    ],
    title="Sistema de Inspeção de Transistores THT",
    description="Projeto Final de Visão Computacional. O sistema detecta: transistores bons, pinos cortados, corpo danificado ou peças fora do lugar."
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://07a4e872a64bf0e9d3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
